TODO: Rerun with both organisms!

In [1]:
from pathlib import Path
import pandas as pd
import re
import sqlite3

In [2]:
conn = sqlite3.connect('../sqlite_backend.db')

In [3]:
#Alternative, for future reference
cursor = conn.cursor()
cursor.execute('SELECT TAXCODE, NAME FROM ORGANISM')
cursor.fetchall()

[(9606, 'Homo sapiens'), (10090, 'Mus musculus')]

Retrieve manually inputted data from database

In [4]:
organism_df = pd.read_sql_query('SELECT * FROM ORGANISM', conn)
organism_df

,TAXCODE,NAME
0,9606,Homo sapiens
1,10090,Mus musculus


In [5]:
taxcode = organism_df.loc[0]['TAXCODE']
taxcode

np.int64(9606)

In [6]:
protease_df = pd.read_sql_query('SELECT * FROM PROTEASE', conn)
protease_df

,PROTEASE_ID,NAME,CLEAVAGE_RULE
0,1,Trypsin,([KR])(?!P)


In [7]:
protease_id = protease_df.loc[0]['PROTEASE_ID']
protease_id

np.int64(1)

In [8]:
cleavage_rule = protease_df.loc[0]['CLEAVAGE_RULE']
cleavage_rule_compiled = re.compile(cleavage_rule)
cleavage_rule_compiled

re.compile(r'([KR])(?!P)', re.UNICODE)

In [9]:
modification_df = pd.read_sql_query('SELECT * FROM MODIFICATION', conn)
modification_df

,MODIFICATION_ID,NAME,SYMBOL,MODIFIABLE_RESIDUES
0,1,Phosphorylation,(ph),STY


In [10]:
modification_id = modification_df.loc[0]['MODIFICATION_ID']
modifiable_residues = modification_df.loc[0]['MODIFIABLE_RESIDUES']

Determine Fasta File using the taxcode

In [11]:
matching_files = list(Path('../resources/').glob(f'uniprot_proteome_w_isoforms_{taxcode}*.fasta'))
if len(matching_files) > 1:
    print(f'''More than one fasta files match! Ambiguous.
    The following files match: {[f.name for f in matching_files]} 
    ''')
else:
    fasta_path = matching_files[0]
    print(fasta_path)

..\resources\uniprot_proteome_w_isoforms_9606_20250829.fasta


In [12]:
#For debug
#fasta_path = Path('../resources/uniprot_fasta_stub.fasta')

In [8]:
def digest_sequence(sequence, cleavage_regex, n_missed_cleavages):
    sequences_wo_missed_cleavages = []
    previous_index = 0
    for match in re.finditer(cleavage_regex, sequence):
        sequences_wo_missed_cleavages.append({
            'seq': sequence[previous_index:match.start()+1],
            'start': previous_index+1,
            'end': match.start()+1
        })
        previous_index = match.start()+1
    #Add final stretch
    sequences_wo_missed_cleavages.append({
        'seq': sequence[previous_index:],
        'start': previous_index+1,
        'end': len(sequence)
    })
    sequences_w_missed_cleavages = []
    for i in range(len(sequences_wo_missed_cleavages)):
        for j in range(n_missed_cleavages+1):
            if i+j < len(sequences_wo_missed_cleavages):
                peptide_seq = "".join([entry['seq'] for entry in sequences_wo_missed_cleavages[i:i+j+1]])
                sequences_w_missed_cleavages.append({
                    'SEQUENCE':peptide_seq, 
                    'START_POSITION':sequences_wo_missed_cleavages[i]['start'] , 
                    'END_POSITION': sequences_wo_missed_cleavages[i+j]['end']})
    
    #Optionally cleave away the N-terminal methionine
    res = []
    for tryptic_peptide in sequences_w_missed_cleavages:
        res.append(tryptic_peptide)
        if tryptic_peptide['START_POSITION'] == 1 and tryptic_peptide['SEQUENCE'][0] == 'M':
            res.append({'SEQUENCE':tryptic_peptide['SEQUENCE'][1:] , 'START_POSITION':2, 'END_POSITION': tryptic_peptide['END_POSITION']})
    return res

In [14]:
def get_or_create_peptide_ids(peptides_of_protein, peptides, protein_to_peptides, current_peptide_id):
    for peptide in peptides_of_protein:
        if (existing_peptide_id:=peptides.get(peptide['SEQUENCE'])):
            peptide['PEPTIDE_ID'] = existing_peptide_id
        else:
            peptide['PEPTIDE_ID'] = current_peptide_id
            peptides[peptide['SEQUENCE']] = current_peptide_id
            current_peptide_id += 1
        peptide['PROTEIN_ID'] = current_protein_id
        #Remove sequence, it is not stored in the protein_to_peptides table
        peptide.pop('SEQUENCE')
        protein_to_peptides.append(peptide)
    return protein_to_peptides, peptides, current_peptide_id

In [15]:
isoform_regex = re.compile(r' Isoform of ([A-Z0-9]+),')
def get_canonical_protein_id(proteins, rest_string):
    isoform_match = re.search(isoform_regex, rest)
    if isoform_match:
        parent_protein = proteins.get(isoform_match.group(1))
        if not parent_protein:
            print(f'Canonical Isoform {isoform_match} not found!')
            return None
        else: 
            return parent_protein['PROTEIN_ID']
    else: 
        return None

In [16]:
gene_name_regex = re.compile(r'GN=([A-Z0-9]+)')
def get_gene_name(rest_string):
    gn_match = re.search(gene_name_regex, rest_string)
    if gn_match:
        return gn_match.group(1)
    else:
        return None

In [17]:
def extract_modified_sites(modified_sites, current_modified_site_id, protein_sequence, protein_id, uniprot_acc, modification_id, modifiable_residues):
    for index, aa in enumerate(protein_sequence):
        if aa in modifiable_residues:
            modified_sites.append({
                'MODIFIED_SITE_ID': current_modified_site_id,
                'PROTEIN_ID': protein_id,
                'MODIFICATION_ID': modification_id,
                'POSITION': index+1,
                'RESIDUE': aa,
                'SITE_IDENTIFIER': f'{uniprot_acc}_{aa}{index+1}'
            })
            current_modified_site_id +=1
    return modified_sites, current_modified_site_id

In [18]:
with open(fasta_path) as fasta_parsed:
    proteins = {}
    peptides = {}
    protein_to_peptides = []
    modified_sites = []
    current_protein_id = 1
    current_peptide_id = 1
    current_modified_site_id = 1
    sequence = ''
    while True:
        line=fasta_parsed.readline().strip()
        if not line or line[0] == '>':
            if sequence:
                #Digest using Trypsine cleavage rules and up to 4 missed cleavages
                peptides_of_protein = digest_sequence(sequence, cleavage_rule_compiled, 4)
                #Get or create peptide ids
                protein_to_peptides, peptides, current_peptide_id = get_or_create_peptide_ids(
                    peptides_of_protein, peptides, protein_to_peptides, current_peptide_id)
                #Extract modified sites
                modified_sites, current_modified_site_id = extract_modified_sites(modified_sites, current_modified_site_id, 
                                                                                  sequence, current_protein_id, 
                                                                                  uniprot, modification_id, modifiable_residues)
                #Check if the protein is a non-canonical isoform
                parent_protein_id = get_canonical_protein_id(proteins, rest)
                #Insert finished protein, maintain as a map with uniprots as keys - this way isoforms can easily find their parent
                proteins[uniprot] = {'PROTEIN_ID': current_protein_id, 'UNIPROT_ACC':uniprot,
                                     'GENE_NAME':gene_name, 'PARENT_PROTEIN_ID': parent_protein_id}
                #Prepare for the next protein
                if current_protein_id % 1000 == 0:
                    print(f'Processed {current_protein_id} proteins')
                current_protein_id += 1
                sequence = ''
            if line:
                db, uniprot, rest = line[1:].split('|')
                gene_name = get_gene_name(rest)
            else:
                break
        else:
            sequence += line 

Processed 100 proteins
Processed 200 proteins
Processed 300 proteins
Processed 400 proteins
Processed 500 proteins
Processed 600 proteins
Processed 700 proteins
Processed 800 proteins
Processed 900 proteins
Processed 1000 proteins
Processed 1100 proteins
Processed 1200 proteins
Processed 1300 proteins
Processed 1400 proteins
Processed 1500 proteins
Processed 1600 proteins
Processed 1700 proteins
Processed 1800 proteins
Processed 1900 proteins
Processed 2000 proteins
Processed 2100 proteins
Processed 2200 proteins
Processed 2300 proteins
Processed 2400 proteins
Processed 2500 proteins
Processed 2600 proteins
Processed 2700 proteins
Processed 2800 proteins
Processed 2900 proteins
Processed 3000 proteins
Processed 3100 proteins
Processed 3200 proteins
Processed 3300 proteins
Processed 3400 proteins
Processed 3500 proteins
Processed 3600 proteins
Processed 3700 proteins
Processed 3800 proteins
Processed 3900 proteins
Processed 4000 proteins
Processed 4100 proteins
Processed 4200 proteins
P

In [19]:
proteins_df = pd.DataFrame(proteins.values())
proteins_df['PARENT_PROTEIN_ID'] = proteins_df['PARENT_PROTEIN_ID'].astype('Int64')
proteins_df['TAXCODE'] = taxcode
proteins_df

,PROTEIN_ID,UNIPROT_ACC,GENE_NAME,PARENT_PROTEIN_ID,TAXCODE
0,1,A0A087X0M5,TRBV18,<NA>,9606
1,2,A6NEH6,TMEM247,<NA>,9606
2,3,A6NIH7,UNC119B,<NA>,9606
3,4,A6NJR5,None,<NA>,9606
4,5,A6NKF7,TMEM278,<NA>,9606
...,...,...,...,...,...
105714,105715,A0A0D9SG52,None,<NA>,9606
105715,105716,A0A1W2PRQ8,None,<NA>,9606
105716,105717,C9J4A7,None,<NA>,9606
105717,105718,G3V3Y1,None,<NA>,9606


In [24]:
proteins_df.to_sql('PROTEIN', conn, if_exists='append', index=False)

105719

In [20]:
peptides_df = pd.DataFrame(peptides.items(), columns=['SEQUENCE','PEPTIDE_ID'])
peptides_df

,SEQUENCE,PEPTIDE_ID
0,MDTR,1
1,MDTRLLCCAVICLLGAGLSNAGVMQNPR,2
2,MDTRLLCCAVICLLGAGLSNAGVMQNPRHLVR,3
3,MDTRLLCCAVICLLGAGLSNAGVMQNPRHLVRR,4
4,MDTRLLCCAVICLLGAGLSNAGVMQNPRHLVRRR,5
...,...,...
6215931,XGSWYK,6215932
6215932,XGSWYKHVK,6215933
6215933,XGSWYKHVKSWWEK,6215934
6215934,XGSWYKHVKSWWEKGK,6215935


In [25]:
peptides_df.to_sql('PEPTIDE', conn, if_exists='append', index=False)

6215936

In [21]:
protein_to_peptides_df = pd.DataFrame(protein_to_peptides)
protein_to_peptides_df['PROTEASE_ID'] = protease_id
protein_to_peptides_df

,START_POSITION,END_POSITION,PEPTIDE_ID,PROTEIN_ID,PROTEASE_ID
0,1,4,1,1,1
1,1,28,2,1,1
2,1,32,3,1,1
3,1,33,4,1,1
4,1,34,5,1,1
...,...,...,...,...,...
23322808,118,124,525308,105719,1
23322809,118,127,525309,105719,1
23322810,123,124,759,105719,1
23322811,123,127,525310,105719,1


In [26]:
protein_to_peptides_df.to_sql('PROTEIN_TO_PEPTIDE', conn, if_exists='append', index=False)

23322813

In [22]:
modified_sites_df = pd.DataFrame(modified_sites)
modified_sites_df

,MODIFIED_SITE_ID,PROTEIN_ID,MODIFICATION_ID,POSITION,RESIDUE,SITE_IDENTIFIER
0,1,1,1,3,T,A0A087X0M5_T3
1,2,1,1,19,S,A0A087X0M5_S19
2,3,1,1,43,S,A0A087X0M5_S43
3,4,1,1,49,S,A0A087X0M5_S49
4,5,1,1,52,Y,A0A087X0M5_Y52
...,...,...,...,...,...,...
7316638,7316639,105719,1,101,T,H0Y8G0_T101
7316639,7316640,105719,1,112,Y,H0Y8G0_Y112
7316640,7316641,105719,1,119,S,H0Y8G0_S119
7316641,7316642,105719,1,120,T,H0Y8G0_T120


In [27]:
modified_sites_df.to_sql('MODIFIED_SITE', conn, if_exists='append', index=False)

7316643